In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime

# Einstellungen
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [ ]:
# Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()

# Pfad zur "Komplette Datenbank" Ebene
db_root = (
    base_dir.parent 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
    / "Komplette_Datenbank"
)

# Alle Unterordner lesen, Ordner mit "neustem" Datum setzen
timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Datenbank-Ordner in {db_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# Komplette Datenbank als CSV
input_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(input_path)

In [ ]:
# Ordner "Preprocessing"
output_root = base_dir / "Preprocessing"
output_root.mkdir(exist_ok=True)

# Zeitstempel erzeugen
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Neuen Ordner mit Zeitstempel erzeugen
output_dir = output_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)

<div class="alert alert-info">
    <b>Preprocessing & Daten-Transformation</b><br>
    - Laden der aktuellsten Datenbank<br>
    - Ausschluss von Metadaten (Koordinaten, IDs, ...) von der Transformation<br>
    - Analyse auf Schiefe (Skewness) der Verteilung<br>
    - Anwendung von Log-Transformation (log1p) bei Bedarf<br>
    - Vergleich Vorher vs. Nachher<br>
    - Speichern der angepassten Daten
</div>

In [ ]:
# Daten laden
df = pd.read_csv(input_path, low_memory=False)

print(f"Datensatz geladen mit {df.shape[0]} Zeilen und {df.shape[1]} Spalten.")
df.head()

In [ ]:
# ----------------- Spalten ausschließen -----------------
# Diese Spalten sollen NICHT transformiert werden, auch wenn sie numerisch sind.
EXCLUDE_COLS = [
    "WGS84_latitude",
    "WGS84_longitude",
    "Database_number",
    "id",                 # Falls vorhanden
    "index"               # Falls vorhanden
]

# Zusätzlich sicherstellen, dass nicht-numerische (die evtl. als Object geladen wurden) ignoriert werden
# Identifikation numerischer Spalten
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Kandidaten für Transformation (Numerisch OHNE Excluded)
transform_candidates = [c for c in numeric_cols if c not in EXCLUDE_COLS]

print(f"{len(transform_candidates)} numerische Spalten kommen für Transformation in Frage.")
print(f"Ausgeschlossen (Metadata/Coords): {len([c for c in numeric_cols if c in EXCLUDE_COLS])}")

In [ ]:
# ----------------- Schiefe berechnen -----------------
skewness = df[transform_candidates].skew().sort_values(ascending=False)

# Filter für stark schiefe Spalten (Threshold > 1.0)
skew_threshold = 1.0
skewed_cols = skewness[abs(skewness) > skew_threshold]

transformed_cols_list = skewed_cols.index.tolist()
unchanged_cols_list = [c for c in df.columns if c not in transformed_cols_list]

print("=== Transformations-Bericht ===")
print(f"Anzahl transformierter Spalten: {len(transformed_cols_list)}")
print(f"Anzahl unveränderter Spalten:   {len(unchanged_cols_list)}")

print("\n--- Transformierte Spalten (Top 10 nach Skewness) ---")
print(skewed_cols.head(10))

print("\n--- Unveränderte Spalten (Beispiele) ---")
print(unchanged_cols_list[:10])

In [ ]:
# ----------------- Transformation & Visualisierung -----------------
df_transformed = df.copy()
applied_transformations = []

for col in transformed_cols_list:
    # Prüfen auf nicht-negative Werte
    if df[col].min() >= 0:
        new_col_name = f"{col}_log"
        df_transformed[new_col_name] = np.log1p(df[col])
        applied_transformations.append(col)
        
        # --- Visualisierung Vorher / Nachher ---
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Original
        sns.histplot(df[col].dropna(), kde=True, bins=30, ax=axes[0], color="skyblue")
        axes[0].set_title(f"Original: {col} (Skew: {skewness[col]:.2f})")
        
        # Log-Transformiert
        # Skewness neu berechnen für Titel
        new_skew = df_transformed[new_col_name].skew()
        sns.histplot(df_transformed[new_col_name].dropna(), kde=True, bins=30, ax=axes[1], color="lightgreen")
        axes[1].set_title(f"Transformiert: {new_col_name} (Skew: {new_skew:.2f})")
        
        plt.tight_layout()
        plt.show()

print(f"\nTransformation abgeschlossen für {len(applied_transformations)} Spalten.")

In [ ]:
# ----------------- Ergebnis speichern -----------------
output_file = output_dir / "Preprocessed_Database.csv"
df_transformed.to_csv(output_file, index=False)

print(f"Fertig! Datei gespeichert unter:\n{output_file}")